In [1]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
from flax import nnx
import jax.tree_util as jtu
import sys
sys.path.insert(0, '.')
# H2分子几何构型
bond_length = 1.4  # 平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

print("="*60)
print("H2分子系统设置")
print("="*60)
print(f"键长: {bond_length} 埃")

# 创建分子对象
mol = gto.M(atom=geometry, basis='STO-3G')

# Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"\nHartree-Fock能量: {E_hf:.8f} Ha")

# FCI计算（参考值）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print(f"\nFCI能级（参考值）:")
print("-"*50)
for i, e in enumerate(E_fcis):
    exc_energy = (e - E_fcis[0]) * 27.2114
    if i == 0:
        print(f"  E{i} (基态)     = {e:.8f} Ha")
    else:
        print(f"  E{i} (第{i}激发态) = {e:.8f} Ha  激发能: {exc_energy:.4f} eV")

# 创建NetKet哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H2分子系统设置
键长: 1.4 埃

Hartree-Fock能量: -0.94148065 Ha

FCI能级（参考值）:
--------------------------------------------------
  E0 (基态)     = -1.01546825 Ha
  E1 (第1激发态) = -0.87542794 Ha  激发能: 3.8107 eV
  E2 (第2激发态) = -0.42938376 Ha  激发能: 15.9482 eV
  E3 (第3激发态) = -0.26922131 Ha  激发能: 20.3064 eV


In [2]:
class SingleStateAnsatz(nnx.Module):
    """单态 Ansatz：适配费米子系统的复数值 FFNN"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_spin_orbitals = n_spin_orbitals
        # 复数值线性层（费米子波函数需相位描述）
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        输入：x = 单组自旋轨道占据态（形状 [n_spin_orbitals,]，如 H₂ 的 [0,1,0,1]）
        输出：复数值波函数值 ψ(x)
        """
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)  # 输出形状 []（标量复数）
    
class NESTotalAnsatz(nnx.Module):
    """NES-VMC 总 Ansatz：K 个单态 Ansatz 的行列式"""
    def __init__(self, n_spin_orbitals: int, n_states: int = 3, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states  # K
        self.n_spin_orbitals = n_spin_orbitals
        
        # 初始化 K 个独立的单态 Ansatz
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(42+i))
            for i in range(n_states)
        ]

    def __call__(self, x_batch: jax.Array) -> tuple[jax.Array, jax.Array]:
        """x_batch.shape = [K, n_spin_orbitals]
        
        输出：
            psi_total : 行列式标量（给 NetKet 采样）
            M_matrix  : K×K 矩阵（给 Eq.17 局部能量计算）
        """
        if x_batch.shape[0] != self.n_states:
            raise ValueError(f"x_batch.shape[0] != {self.n_states}")
        K = self.n_states
        M = []
        for i in range(K):
            for j in range(K):
                # M[i,j] = ψ_j(x_i)
                psi_i_xj = self.single_ansatz_list[j](x_batch[i])
                M.append(psi_i_xj)
        
        # 构建 K×K 矩阵
        M = jnp.stack(M, axis=0)
        M_matrix = M.reshape(K, K)  # 这就是论文里的 Ψ 矩阵！
        # 计算行列式
        psi_total = jnp.linalg.det(M_matrix)
        return psi_total, M_matrix

In [5]:
def Ham_psi(ha:nk.operator.DiscreteOperator,model,x:jnp.array):
    x_primes, mels = ha.get_conn(x)
    psi_values = jax.vmap(model)(x_primes)
    H_psi_x = jnp.sum(mels*psi_values)
    return H_psi_x

def Ham_Psi(ha:nk.operator.DiscreteOperator,total_ansatz,x:jnp.array):
    k = total_ansatz.n_states
    if x.shape[0] != k:
        raise ValueError(f"Input array must have shape ({k},) but got shape {x.shape}")
    H_psi_x_i = []
    for i in range(k):
        tmp = []
        for j in range(k):
            ele = Ham_psi(ha,model=total_ansatz.single_ansatz_list[j],x=x[i])
            tmp.append(ele)
            
    
        H_psi_x_i.append(tmp)
    
    HPsi = jnp.array(H_psi_x_i).reshape(k,k)    
    return HPsi            